# VolumeBar Schema Reference

## Table of Contents

1. [Configuration & Identifiers](#1-configuration--identifiers)
2. [Metadata Fields](#2-metadata-fields)
3. [Order & Volume Statistics (Ambiguous/Total)](#3-order--volume-statistics-ambiguoustotal)
4. [Active Variables](#4-active-variables)
    - 4.1 [Active Order Counts](#41-active-order-counts)
    - 4.2 [Active Volumes](#42-active-volumes)
    - 4.3 [Active Imbalance Metrics](#43-active-imbalance-metrics)
    - 4.4 [Active Link Function Transforms](#44-active-link-function-transforms)
    - 4.5 [Active Price Metrics (VWAP)](#45-active-price-metrics-vwap)
    - 4.6 [Active Weighted Midpoints](#46-active-weighted-midpoints)
    - 4.7 [Active Price Range Metrics](#47-active-price-range-metrics)
    - 4.8 [Active Pace Metrics](#48-active-pace-metrics)
    - 4.9 [Active N-Side Inferred Classification](#49-active-n-side-inferred-classification)
5. [Adjusted Variables (Active + N-Side Inferred)](#5-adjusted-variables-active--n-side-inferred)
    - 5.1 [Adjusted Volumes](#51-adjusted-volumes)
    - 5.2 [Adjusted Imbalance Metrics](#52-adjusted-imbalance-metrics)
    - 5.3 [Adjusted Link Function Transforms](#53-adjusted-link-function-transforms)
    - 5.4 [Adjusted Price Metrics (VWAP)](#54-adjusted-price-metrics-vwap)
    - 5.5 [Adjusted Weighted Midpoints](#55-adjusted-weighted-midpoints)
6. [Passive Variables](#6-passive-variables)
    - 6.1 [Passive Midprice Metrics](#61-passive-midprice-metrics)
    - 6.2 [Passive Normalized and CDF Metrics](#62-passive-normalized-and-cdf-metrics)
    - 6.3 [Passive Volume Classification](#63-passive-volume-classification)
    - 6.4 [Passive Imbalance Metrics](#64-passive-imbalance-metrics)
    - 6.5 [Passive Link Function Transforms](#65-passive-link-function-transforms)
7. [Temporal Metrics](#7-temporal-metrics)
8. [Instrument Tracking](#8-instrument-tracking)
9. [Divergence Metrics](#9-divergence-metrics)
10. [Derived Indicator](#10-derived-indicator)
11. [Delta Calculations](#11-delta-calculations)
    - 11.1 [Ambiguous/Total Deltas](#111-ambiguoustotal-deltas)
    - 11.2 [Active Deltas](#112-active-deltas)
    - 11.3 [Adjusted Deltas](#113-adjusted-deltas)
    - 11.4 [Passive Deltas](#114-passive-deltas)
    - 11.5 [Temporal Deltas](#115-temporal-deltas)
    - 11.6 [Divergence Deltas](#116-divergence-deltas)
12. [Calculation Order in _calculate_statistics()](#12-calculation-order-in-_calculate_statistics)
13. [Edge Case Handling Summary](#13-edge-case-handling-summary)

---

## 1. Configuration & Identifiers

|Variable|Type|Description|Edge Case Fill|
|---|---|---|---|
|`id`|`uint32`|Unique identifier for the volume bar. Initially 0, updated when previous bar info available.|0|
|`bar_volume_size`|`uint32`|Target volume threshold that defines bar completion.|—|

---

## 2. Metadata Fields

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`gap_return`|`bool`|Was there a large time gap during this bar?|From `self._gap_return`|False|
|`max_time_gap_ns`|`uint64`|Largest gap between consecutive timestamps in nanoseconds|`np.max(np.diff(central_timestamps))` if `gap_return` else 0|0|
|`contains_oversized_order`|`bool`|Did bar encounter an oversized order?|From `self._oversized_order`|False|
|`has_resized`|`bool`|Did arrays resize during collection?|From `self._has_resized`|False|

---

## 3. Order & Volume Statistics (Ambiguous/Total)

These metrics include all order types (A + B + N) and remain unprefixed.

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`order_count`|`uint32`|Total orders in bar (A + B + N)|`a_idx + b_idx + n_idx`|0|
|`order_splits`|`uint32`|Count of orders spanning multiple bars|`sum(a_is_continuation) + sum(b_is_continuation) + sum(n_is_continuation)`|0|
|`volume_total`|`uint32`|Total volume transacted (A + B + N)|`active_volume_buy + active_volume_sell + active_volume_none`|0|
|`bar_complete`|`bool`|Did bar reach target volume?|`volume_total == bar_volume_size`|—|

---

## 4. Active Variables

Active variables are derived from trades where the aggressor side is known (FIX tag 5796).

**Terminology:**

- `active_buy_*` = Aggressor lifted the offer (buyer crossed spread)
- `active_sell_*` = Aggressor hit the bid (seller crossed spread)
- `active_none_*` = Aggressor side unknown (N-side)

### 4.1 Active Order Counts

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_order_count_buy`|`uint32`|Count of buy aggressor orders|`b_buy_aggressor_idx`|0|
|`active_order_count_sell`|`uint32`|Count of sell aggressor orders|`a_sell_aggressor_idx`|0|
|`active_order_count_none`|`uint32`|Count of N-side orders|`n_none_aggressor_idx`|0|

### 4.2 Active Volumes

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_volume_buy`|`uint32`|Volume from buy aggressor orders|`sum(b_buy_aggressor_size_contributed[:b_idx])`|0|
|`active_volume_sell`|`uint32`|Volume from sell aggressor orders|`sum(a_sell_aggressor_size_contributed[:a_idx])`|0|
|`active_volume_none`|`uint32`|Volume from N-side orders|`sum(n_none_aggressor_size_contributed[:n_idx])`|0|

### 4.3 Active Imbalance Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_imbalance_signed`|`int64`|Net directional imbalance (buy - sell)|`active_volume_buy - active_volume_sell`|0|
|`active_imbalance_abs`|`uint64`|Absolute magnitude of imbalance|`abs(active_imbalance_signed)`|0|
|`active_imbalance_signed_ratio`|`float64`|Signed imbalance / total volume. Range: [-1, 1]|`active_imbalance_signed / volume_total`|NaN if volume_total = 0|
|`active_imbalance_abs_ratio`|`float64`|Absolute imbalance / total volume. Range: [0, 1]|`active_imbalance_abs / volume_total`|NaN if volume_total = 0|
|`active_imbalance_buy_ratio`|`float64`|Buy volume / total volume. Range: [0, 1]|`active_volume_buy / volume_total`|NaN if volume_total = 0|

### 4.4 Active Link Function Transforms

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_imbalance_signed_ratio_atanh`|`float64`|Arctanh transform of signed ratio. Range: (-∞, +∞)|`arctanh(clip(active_imbalance_signed_ratio, -0.9999, 0.9999))`|NaN if ratio is NaN|
|`active_imbalance_buy_ratio_logit`|`float64`|Logit transform of buy ratio. Range: (-∞, +∞)|`log(r / (1 - r))` where r = clip(buy_ratio, 0.0001, 0.9999)|NaN if ratio is NaN|
|`active_imbalance_abs_ratio_logit`|`float64`|Logit transform of abs ratio. Range: (-∞, +∞)|`log(r / (1 - r))` where r = clip(abs_ratio, 0.0001, 0.9999)|NaN if ratio is NaN|

### 4.5 Active Price Metrics (VWAP)

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_buy_vwap`|`float64`|Volume-weighted average price of buy aggressor trades|`sum(buy_prices × buy_volumes) / sum(buy_volumes)`|NaN if no buy trades|
|`active_sell_vwap`|`float64`|Volume-weighted average price of sell aggressor trades|`sum(sell_prices × sell_volumes) / sum(sell_volumes)`|NaN if no sell trades|
|`active_none_vwap`|`float64`|Volume-weighted average price of N-side trades|`sum(none_prices × none_volumes) / sum(none_volumes)`|NaN if no N-side trades|
|`active_spread_vwap`|`float64`|Difference between buy and sell VWAPs|`active_buy_vwap - active_sell_vwap`|NaN if either VWAP is NaN|
|`active_midpoint_vwap`|`float64`|Simple average of buy and sell VWAPs|`(active_buy_vwap + active_sell_vwap) * 0.5`|NaN if either VWAP is NaN|

### 4.6 Active Weighted Midpoints

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_mid_imbalance_weighted`|`float64`|Midpoint adjusted for order flow imbalance (linear tilt)|`active_midpoint_vwap + (active_spread_vwap * 0.5 * clip(active_imbalance_signed_ratio, -1, 1))`|NaN if midpoint or spread is NaN|
|`active_mid_flow_weighted`|`float64`|Center of mass of executed trades (same-side weighting)|`(active_sell_vwap × active_volume_sell + active_buy_vwap × active_volume_buy) / (active_volume_buy + active_volume_sell)`|NaN if no A/B trades|
|`active_mid_aggressor_weighted`|`float64`|Microprice analogue (opposite-side weighting)|`(active_sell_vwap × active_volume_buy + active_buy_vwap × active_volume_sell) / (active_volume_buy + active_volume_sell)`|NaN if no A/B trades|

### 4.7 Active Price Range Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_buy_price_min`|`float64`|Minimum price of buy aggressor trades|`min(buy_prices[:b_idx])`|NaN if no buy trades|
|`active_buy_price_max`|`float64`|Maximum price of buy aggressor trades|`max(buy_prices[:b_idx])`|NaN if no buy trades|
|`active_buy_price_range`|`float64`|Price range of buy aggressor trades|`active_buy_price_max - active_buy_price_min`|NaN if no buy trades|
|`active_sell_price_min`|`float64`|Minimum price of sell aggressor trades|`min(sell_prices[:a_idx])`|NaN if no sell trades|
|`active_sell_price_max`|`float64`|Maximum price of sell aggressor trades|`max(sell_prices[:a_idx])`|NaN if no sell trades|
|`active_sell_price_range`|`float64`|Price range of sell aggressor trades|`active_sell_price_max - active_sell_price_min`|NaN if no sell trades|
|`active_none_price_min`|`float64`|Minimum price of N-side trades|`min(none_prices[:n_idx])`|NaN if no N-side trades|
|`active_none_price_max`|`float64`|Maximum price of N-side trades|`max(none_prices[:n_idx])`|NaN if no N-side trades|
|`active_none_price_range`|`float64`|Price range of N-side trades|`active_none_price_max - active_none_price_min`|NaN if no N-side trades|

### 4.8 Active Pace Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_buy_pace`|`float64`|Buy volume flow rate (contracts per nanosecond)|`active_volume_buy / time_elapsed_ns`|NaN if time_elapsed_ns = 0|
|`active_sell_pace`|`float64`|Sell volume flow rate (contracts per nanosecond)|`active_volume_sell / time_elapsed_ns`|NaN if time_elapsed_ns = 0|
|`active_buy_pace_log`|`float64`|Log transform of buy pace|`log(active_buy_pace)`|NaN if pace ≤ 0 or NaN|
|`active_sell_pace_log`|`float64`|Log transform of sell pace|`log(active_sell_pace)`|NaN if pace ≤ 0 or NaN|

### 4.9 Active N-Side Inferred Classification

N-side trades are classified as "likely buy" or "likely sell" by comparing trade price to a reference price derived from active VWAPs.

**Reference Price Logic:**

```python
if active_buy_vwap is not NaN and active_sell_vwap is not NaN:
    reference = (active_buy_vwap + active_sell_vwap) * 0.5
elif active_buy_vwap is not NaN:
    reference = active_buy_vwap
elif active_sell_vwap is not NaN:
    reference = active_sell_vwap
elif previous["active_midpoint_vwap"] is not NaN:
    reference = previous["active_midpoint_vwap"]  # FALLBACK
elif previous["passive_midprice"] is not NaN:
    reference = previous["passive_midprice"]  # FALLBACK
else:
    reference = NaN  # Cannot classify
```

**SPECIAL CASE - PREVIOUS BAR FALLBACK:** When the entire bar consists of N-side orders only (no A-side or B-side), we cannot derive a reference price from current bar data. This typically occurs when an oversized N-side order (e.g., auction, cross, or spread trade) fills the entire volume bar. In this case, we fall back to the previous bar's midpoint to enable classification. This is logged as CRITICAL because it indicates unusual market conditions that may affect metric reliability.

**Classification Logic (per N-side trade):**

```python
if trade_price > reference:
    inferred_buy_volume += trade_size
else:
    inferred_sell_volume += trade_size
```

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`active_none_inferred_buy_volume`|`float64`|N-side volume inferred as buy aggressor|Sum of N-side sizes where price > reference|0.0 if no N-side trades or reference is NaN|
|`active_none_inferred_sell_volume`|`float64`|N-side volume inferred as sell aggressor|Sum of N-side sizes where price ≤ reference|0.0 if no N-side trades or reference is NaN|
|`active_none_inferred_buy_vwap`|`float64`|VWAP of N-side trades inferred as buy|`sum(inferred_buy_prices × inferred_buy_volumes) / sum(inferred_buy_volumes)`|NaN if no inferred buy volume|
|`active_none_inferred_sell_vwap`|`float64`|VWAP of N-side trades inferred as sell|`sum(inferred_sell_prices × inferred_sell_volumes) / sum(inferred_sell_volumes)`|NaN if no inferred sell volume|

---

## 5. Adjusted Variables (Active + N-Side Inferred)

Adjusted variables incorporate N-side inferred classifications into the active metrics. This provides a more complete picture of order flow by including trades where the aggressor side was unknown but could be inferred from price relative to a reference.

### 5.1 Adjusted Volumes

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`adjusted_volume_buy`|`float64`|Active buy + inferred N-side buy volume|`active_volume_buy + active_none_inferred_buy_volume`|0.0 if both components are 0|
|`adjusted_volume_sell`|`float64`|Active sell + inferred N-side sell volume|`active_volume_sell + active_none_inferred_sell_volume`|0.0 if both components are 0|

### 5.2 Adjusted Imbalance Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`adjusted_imbalance_signed`|`float64`|Net directional imbalance (adjusted buy - sell)|`adjusted_volume_buy - adjusted_volume_sell`|0.0|
|`adjusted_imbalance_abs`|`float64`|Absolute magnitude of adjusted imbalance|`abs(adjusted_imbalance_signed)`|0.0|
|`adjusted_imbalance_signed_ratio`|`float64`|Signed imbalance / total volume. Range: [-1, 1]|`adjusted_imbalance_signed / volume_total`|NaN if volume_total = 0|
|`adjusted_imbalance_abs_ratio`|`float64`|Absolute imbalance / total volume. Range: [0, 1]|`adjusted_imbalance_abs / volume_total`|NaN if volume_total = 0|
|`adjusted_imbalance_buy_ratio`|`float64`|Adjusted buy volume / total volume. Range: [0, 1]|`adjusted_volume_buy / volume_total`|NaN if volume_total = 0|

### 5.3 Adjusted Link Function Transforms

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`adjusted_imbalance_signed_ratio_atanh`|`float64`|Arctanh transform of signed ratio. Range: (-∞, +∞)|`arctanh(clip(adjusted_imbalance_signed_ratio, -0.9999, 0.9999))`|NaN if ratio is NaN|
|`adjusted_imbalance_buy_ratio_logit`|`float64`|Logit transform of buy ratio. Range: (-∞, +∞)|`log(r / (1 - r))` where r = clip(buy_ratio, 0.0001, 0.9999)|NaN if ratio is NaN|
|`adjusted_imbalance_abs_ratio_logit`|`float64`|Logit transform of abs ratio. Range: (-∞, +∞)|`log(r / (1 - r))` where r = clip(abs_ratio, 0.0001, 0.9999)|NaN if ratio is NaN|

### 5.4 Adjusted Price Metrics (VWAP)

Adjusted VWAPs combine active VWAPs with inferred N-side VWAPs, weighted by their respective volumes.

**Formula:**

```python
adjusted_buy_vwap = (active_buy_vwap * active_volume_buy + 
                    active_none_inferred_buy_vwap * active_none_inferred_buy_volume) /
                   adjusted_volume_buy
```

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`adjusted_buy_vwap`|`float64`|Weighted average of active buy VWAP and inferred N-side buy VWAP|See formula above|NaN if adjusted_volume_buy = 0|
|`adjusted_sell_vwap`|`float64`|Weighted average of active sell VWAP and inferred N-side sell VWAP|See formula above|NaN if adjusted_volume_sell = 0|
|`adjusted_spread_vwap`|`float64`|Difference between adjusted buy and sell VWAPs|`adjusted_buy_vwap - adjusted_sell_vwap`|NaN if either VWAP is NaN|
|`adjusted_midpoint_vwap`|`float64`|Simple average of adjusted buy and sell VWAPs|`(adjusted_buy_vwap + adjusted_sell_vwap) * 0.5`|NaN if either VWAP is NaN|

### 5.5 Adjusted Weighted Midpoints

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`adjusted_mid_imbalance_weighted`|`float64`|Adjusted midpoint with imbalance tilt|`adjusted_midpoint_vwap + (adjusted_spread_vwap * 0.5 * clip(adjusted_imbalance_signed_ratio, -1, 1))`|NaN if midpoint or spread is NaN|
|`adjusted_mid_flow_weighted`|`float64`|Center of mass using adjusted volumes (same-side)|`(adjusted_sell_vwap × adjusted_volume_sell + adjusted_buy_vwap × adjusted_volume_buy) / (adjusted_volume_buy + adjusted_volume_sell)`|NaN if total adjusted volume = 0|
|`adjusted_mid_aggressor_weighted`|`float64`|Predictive midpoint using adjusted volumes (opposite-side)|`(adjusted_sell_vwap × adjusted_volume_buy + adjusted_buy_vwap × adjusted_volume_sell) / (adjusted_volume_buy + adjusted_volume_sell)`|NaN if total adjusted volume = 0|

---

## 6. Passive Variables

Passive variables treat all trades equally, ignoring the aggressor flag. Buy/sell classification is inferred from price movement using a CDF-based probabilistic model.

**Calculation Dependencies:**

- `DeltasDistributionLookup.std()` → Global σ for normalizing price changes
- `ClassifierDistributionLookup.cdf()` → Standard normal CDF for probability conversion

### 6.1 Passive Midprice Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`passive_midprice`|`float64`|VWAP of all trades (A + B + N combined)|`sum(all_prices × all_volumes) / sum(all_volumes)`|NaN if no trades|
|`passive_midprice_delta_price`|`float64`|Price change vs previous bar|`passive_midprice[i] - passive_midprice[i-1]`|NaN if first bar|
|`passive_midprice_delta_percent`|`float64`|Percent return (decimal form)|`(passive_midprice[i] - passive_midprice[i-1]) / passive_midprice[i-1]`|NaN if first bar or prev = 0|
|`passive_midprice_delta_log`|`float64`|Log return|`log(passive_midprice[i] / passive_midprice[i-1])`|NaN if first bar or prev ≤ 0|

### 6.2 Passive Normalized and CDF Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`passive_midprice_delta_normalized`|`float64`|Price delta scaled by global σ (z-score)|`passive_midprice_delta_price / DeltasDistributionLookup.std()`|NaN if delta is NaN or σ = 0|
|`passive_midprice_delta_cdf`|`float64`|CDF probability score. Range: [0, 1]|`ClassifierDistributionLookup.cdf(passive_midprice_delta_normalized)`|0.5 if normalized = 0; NaN if normalized is NaN|

**Interpretation:** The CDF value represents the probability that the price movement was driven by buying pressure. Values close to 1 indicate strong buying; values close to 0 indicate strong selling; 0.5 indicates neutral.

### 6.3 Passive Volume Classification

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`passive_buy_volume`|`float64`|Inferred buy volume (fractional allowed)|`passive_midprice_delta_cdf × volume_total`|0.0 if CDF is NaN|
|`passive_sell_volume`|`float64`|Inferred sell volume (fractional allowed)|`volume_total - passive_buy_volume`|0.0 if CDF is NaN|

**Note:** Fractional volumes are intentional. Errors average out over time, and fractional representation preserves probabilistic information.

### 6.4 Passive Imbalance Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`passive_imbalance_signed`|`float64`|Inferred net directional imbalance|`passive_buy_volume - passive_sell_volume`|0.0 if volumes are 0|
|`passive_imbalance_abs`|`float64`|Absolute magnitude of inferred imbalance|`abs(passive_imbalance_signed)`|0.0|
|`passive_imbalance_signed_ratio`|`float64`|Signed imbalance / total volume. Range: [-1, 1]|`passive_imbalance_signed / volume_total`|NaN if volume_total = 0|
|`passive_imbalance_abs_ratio`|`float64`|Absolute imbalance / total volume. Range: [0, 1]|`passive_imbalance_abs / volume_total`|NaN if volume_total = 0|
|`passive_imbalance_buy_ratio`|`float64`|Inferred buy volume / total volume. Range: [0, 1]|`passive_buy_volume / volume_total`|NaN if volume_total = 0|

### 6.5 Passive Link Function Transforms

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`passive_imbalance_signed_ratio_atanh`|`float64`|Arctanh transform of signed ratio|`arctanh(clip(passive_imbalance_signed_ratio, -0.9999, 0.9999))`|NaN if ratio is NaN|
|`passive_imbalance_buy_ratio_logit`|`float64`|Logit transform of buy ratio|`log(r / (1 - r))` where r = clip(buy_ratio, 0.0001, 0.9999)|NaN if ratio is NaN|
|`passive_imbalance_abs_ratio_logit`|`float64`|Logit transform of abs ratio|`log(r / (1 - r))` where r = clip(abs_ratio, 0.0001, 0.9999)|NaN if ratio is NaN|

---

## 7. Temporal Metrics

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`start_ts_ns`|`uint64`|Earliest timestamp in bar (nanoseconds)|`min(central_timestamps[:central_timestamp_idx])`|0 if no timestamps|
|`end_ts_ns`|`uint64`|Latest timestamp in bar (nanoseconds)|`max(central_timestamps[:central_timestamp_idx])`|0 if no timestamps|
|`time_elapsed_ns`|`uint64`|Duration of bar (nanoseconds)|`end_ts_ns - start_ts_ns`|0|
|`pace_of_contracts_traded`|`float64`|Order flow velocity (contracts per nanosecond)|`volume_total / time_elapsed_ns`|NaN if time_elapsed_ns = 0|
|`time_elapsed_ns_log`|`float64`|Log transform of time elapsed|`log(time_elapsed_ns)`|NaN if time_elapsed_ns = 0|
|`pace_of_contracts_traded_log`|`float64`|Log transform of order pace|`log(pace_of_contracts_traded)`|NaN if pace ≤ 0 or NaN|

---

## 8. Instrument Tracking

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`contract_roll`|`bool`|Multiple unique instrument IDs in bar?|`len(unique(all_instrument_ids)) > 1`|False|
|`latest_instrument_id`|`uint32`|Instrument ID of most recent order|Instrument ID at `argmax(all_timestamps)`|0 if no orders|

---

## 9. Divergence Metrics

Divergence metrics compare active (aggressor-known) classification to passive (price-inferred) classification.

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`divergence_buy_volume`|`float64`|Difference in buy volume classification|`active_volume_buy - passive_buy_volume`|NaN if either is NaN|
|`divergence_sell_volume`|`float64`|Difference in sell volume classification|`active_volume_sell - passive_sell_volume`|NaN if either is NaN|
|`divergence_imbalance_signed`|`float64`|Difference in signed imbalance|`active_imbalance_signed - passive_imbalance_signed`|NaN if either is NaN|
|`divergence_imbalance_signed_ratio`|`float64`|Difference in signed imbalance ratio|`active_imbalance_signed_ratio - passive_imbalance_signed_ratio`|NaN if either is NaN|
|`divergence_buy_ratio`|`float64`|Difference in buy ratio|`active_imbalance_buy_ratio - passive_imbalance_buy_ratio`|NaN if either is NaN|

**Interpretation:**

- **Positive divergence_buy_volume:** Active detected more buying than passive inferred. Could indicate large buy orders absorbed without price impact, or spread dynamics not captured by midprice.
- **Negative divergence_buy_volume:** Active detected less buying than passive inferred. Could indicate price moved up despite balanced or sell-heavy flow.
- **Large absolute divergence:** Signals potential predictive opportunity or liquidity dynamics worth investigating.

---

## 10. Derived Indicator

|Variable|Type|Description|Calculation|
|---|---|---|---|
|`derived_price_direction_positive`|`bool`|Upward price movement indicator|`active_midpoint_vwap[i] > active_midpoint_vwap[i-1]`|

---

## 11. Delta Calculations

All deltas compare current bar [i] to previous bar [i-1]. First bar deltas are NaN.

**Delta Types:**

- Raw delta: `value[i] - value[i-1]`
- Percent delta (decimal): `(value[i] - value[i-1]) / value[i-1]`
- Log delta: `log((value[i] + ε) / (value[i-1] + ε))` where ε is a small constant for numerical stability

**Edge Case Handling:**

- Percent delta: NaN if previous value is 0
- Log delta: Uses EPSILON for numerical stability; NaN propagates from NaN inputs

### 11.1 Ambiguous/Total Deltas

#### Order Count Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_order_count`|`int32`|NaN if first bar|
|`delta_order_count_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_order_count_log`|`float64`|NaN if first bar or prev ≤ 0|

#### Order Splits Delta

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_order_splits`|`int32`|NaN if first bar|

#### Volume Total Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_volume_total`|`int32`|NaN if first bar|
|`delta_volume_total_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_volume_total_log`|`float64`|NaN if first bar or prev ≤ 0|

### 11.2 Active Deltas

#### Order Count Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_active_order_count_buy`|`int32`|NaN if first bar|
|`delta_active_order_count_buy_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_active_order_count_buy_log`|`float64`|NaN if first bar or prev ≤ 0|
|`delta_active_order_count_sell`|`int32`|NaN if first bar|
|`delta_active_order_count_sell_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_active_order_count_sell_log`|`float64`|NaN if first bar or prev ≤ 0|
|`delta_active_order_count_none`|`int32`|NaN if first bar|
|`delta_active_order_count_none_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_active_order_count_none_log`|`float64`|NaN if first bar or prev ≤ 0|

#### Volume Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_active_volume_buy`|`int32`|NaN if first bar|
|`delta_active_volume_buy_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_active_volume_buy_log`|`float64`|NaN if first bar or prev ≤ 0|
|`delta_active_volume_sell`|`int32`|NaN if first bar|
|`delta_active_volume_sell_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_active_volume_sell_log`|`float64`|NaN if first bar or prev ≤ 0|
|`delta_active_volume_none`|`int32`|NaN if first bar|
|`delta_active_volume_none_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_active_volume_none_log`|`float64`|NaN if first bar or prev ≤ 0|

#### Imbalance Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_active_imbalance_signed`|`int64`|NaN if first bar|
|`delta_active_imbalance_abs`|`int64`|NaN if first bar|
|`delta_active_imbalance_signed_ratio`|`float64`|NaN if first bar|
|`delta_active_imbalance_abs_ratio`|`float64`|NaN if first bar|
|`delta_active_imbalance_buy_ratio`|`float64`|NaN if first bar|

#### Link Function Transform Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_active_imbalance_signed_ratio_atanh`|`float64`|NaN if first bar|
|`delta_active_imbalance_buy_ratio_logit`|`float64`|NaN if first bar|
|`delta_active_imbalance_abs_ratio_logit`|`float64`|NaN if first bar|

#### Price Deltas (VWAP)

For each price field, three delta variants:

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`active_buy_vwap`|`delta_active_buy_vwap`|`delta_active_buy_vwap_pct`|`delta_active_buy_vwap_log`|
|`active_sell_vwap`|`delta_active_sell_vwap`|`delta_active_sell_vwap_pct`|`delta_active_sell_vwap_log`|
|`active_none_vwap`|`delta_active_none_vwap`|`delta_active_none_vwap_pct`|`delta_active_none_vwap_log`|
|`active_spread_vwap`|`delta_active_spread_vwap`|`delta_active_spread_vwap_pct`|`delta_active_spread_vwap_log`|
|`active_midpoint_vwap`|`delta_active_midpoint_vwap`|`delta_active_midpoint_vwap_pct`|`delta_active_midpoint_vwap_log`|

#### Weighted Midpoint Deltas

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`active_mid_imbalance_weighted`|`delta_active_mid_imbalance_weighted`|`delta_active_mid_imbalance_weighted_pct`|`delta_active_mid_imbalance_weighted_log`|
|`active_mid_flow_weighted`|`delta_active_mid_flow_weighted`|`delta_active_mid_flow_weighted_pct`|`delta_active_mid_flow_weighted_log`|
|`active_mid_aggressor_weighted`|`delta_active_mid_aggressor_weighted`|`delta_active_mid_aggressor_weighted_pct`|`delta_active_mid_aggressor_weighted_log`|

#### Price Range Deltas

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`active_buy_price_min`|`delta_active_buy_price_min`|`delta_active_buy_price_min_pct`|`delta_active_buy_price_min_log`|
|`active_buy_price_max`|`delta_active_buy_price_max`|`delta_active_buy_price_max_pct`|`delta_active_buy_price_max_log`|
|`active_buy_price_range`|`delta_active_buy_price_range`|`delta_active_buy_price_range_pct`|`delta_active_buy_price_range_log`|
|`active_sell_price_min`|`delta_active_sell_price_min`|`delta_active_sell_price_min_pct`|`delta_active_sell_price_min_log`|
|`active_sell_price_max`|`delta_active_sell_price_max`|`delta_active_sell_price_max_pct`|`delta_active_sell_price_max_log`|
|`active_sell_price_range`|`delta_active_sell_price_range`|`delta_active_sell_price_range_pct`|`delta_active_sell_price_range_log`|
|`active_none_price_min`|`delta_active_none_price_min`|`delta_active_none_price_min_pct`|`delta_active_none_price_min_log`|
|`active_none_price_max`|`delta_active_none_price_max`|`delta_active_none_price_max_pct`|`delta_active_none_price_max_log`|
|`active_none_price_range`|`delta_active_none_price_range`|`delta_active_none_price_range_pct`|`delta_active_none_price_range_log`|

#### Pace Deltas

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`active_buy_pace`|`delta_active_buy_pace`|`delta_active_buy_pace_pct`|`delta_active_buy_pace_log`|
|`active_sell_pace`|`delta_active_sell_pace`|`delta_active_sell_pace_pct`|`delta_active_sell_pace_log`|

#### Pace Log Base Deltas (delta of base log field)

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_active_buy_pace_log_base`|`float64`|NaN if first bar|
|`delta_active_sell_pace_log_base`|`float64`|NaN if first bar|

#### N-Side Inferred Volume Deltas

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`active_none_inferred_buy_volume`|`delta_active_none_inferred_buy_volume`|`delta_active_none_inferred_buy_volume_pct`|`delta_active_none_inferred_buy_volume_log`|
|`active_none_inferred_sell_volume`|`delta_active_none_inferred_sell_volume`|`delta_active_none_inferred_sell_volume_pct`|`delta_active_none_inferred_sell_volume_log`|

#### N-Side Inferred VWAP Deltas

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`active_none_inferred_buy_vwap`|`delta_active_none_inferred_buy_vwap`|`delta_active_none_inferred_buy_vwap_pct`|`delta_active_none_inferred_buy_vwap_log`|
|`active_none_inferred_sell_vwap`|`delta_active_none_inferred_sell_vwap`|`delta_active_none_inferred_sell_vwap_pct`|`delta_active_none_inferred_sell_vwap_log`|

### 11.3 Adjusted Deltas

#### Volume Deltas

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`adjusted_volume_buy`|`delta_adjusted_volume_buy`|`delta_adjusted_volume_buy_pct`|`delta_adjusted_volume_buy_log`|
|`adjusted_volume_sell`|`delta_adjusted_volume_sell`|`delta_adjusted_volume_sell_pct`|`delta_adjusted_volume_sell_log`|

#### Imbalance Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_adjusted_imbalance_signed`|`float64`|NaN if first bar|
|`delta_adjusted_imbalance_abs`|`float64`|NaN if first bar|
|`delta_adjusted_imbalance_signed_ratio`|`float64`|NaN if first bar|
|`delta_adjusted_imbalance_abs_ratio`|`float64`|NaN if first bar|
|`delta_adjusted_imbalance_buy_ratio`|`float64`|NaN if first bar|

#### Link Function Transform Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_adjusted_imbalance_signed_ratio_atanh`|`float64`|NaN if first bar|
|`delta_adjusted_imbalance_buy_ratio_logit`|`float64`|NaN if first bar|
|`delta_adjusted_imbalance_abs_ratio_logit`|`float64`|NaN if first bar|

#### Price Deltas (VWAP)

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`adjusted_buy_vwap`|`delta_adjusted_buy_vwap`|`delta_adjusted_buy_vwap_pct`|`delta_adjusted_buy_vwap_log`|
|`adjusted_sell_vwap`|`delta_adjusted_sell_vwap`|`delta_adjusted_sell_vwap_pct`|`delta_adjusted_sell_vwap_log`|
|`adjusted_spread_vwap`|`delta_adjusted_spread_vwap`|`delta_adjusted_spread_vwap_pct`|`delta_adjusted_spread_vwap_log`|
|`adjusted_midpoint_vwap`|`delta_adjusted_midpoint_vwap`|`delta_adjusted_midpoint_vwap_pct`|`delta_adjusted_midpoint_vwap_log`|

#### Weighted Midpoint Deltas

|Base Variable|Delta|Delta Pct|Delta Log|
|---|---|---|---|
|`adjusted_mid_imbalance_weighted`|`delta_adjusted_mid_imbalance_weighted`|`delta_adjusted_mid_imbalance_weighted_pct`|`delta_adjusted_mid_imbalance_weighted_log`|
|`adjusted_mid_flow_weighted`|`delta_adjusted_mid_flow_weighted`|`delta_adjusted_mid_flow_weighted_pct`|`delta_adjusted_mid_flow_weighted_log`|
|`adjusted_mid_aggressor_weighted`|`delta_adjusted_mid_aggressor_weighted`|`delta_adjusted_mid_aggressor_weighted_pct`|`delta_adjusted_mid_aggressor_weighted_log`|

### 11.4 Passive Deltas

#### Midprice First-Order Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_passive_midprice`|`float64`|NaN if first bar|
|`delta_passive_midprice_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_passive_midprice_log`|`float64`|NaN if first bar or prev ≤ 0|

#### Midprice Second-Order Deltas (Acceleration)

|Variable|Type|Description|Calculation|Edge Case Fill|
|---|---|---|---|---|
|`delta_passive_midprice_delta_price`|`float64`|Change in price delta|`delta_price[i] - delta_price[i-1]`|NaN if first/second bar|
|`delta_passive_midprice_delta_price_pct`|`float64`|Percent change of delta (decimal)|`(delta_price[i] - delta_price[i-1]) / delta_price[i-1]`|NaN if prev delta = 0|
|`delta_passive_midprice_delta_price_log`|`float64`|Log change of delta|`log((delta_price[i] + ε) / (delta_price[i-1] + ε))`|NaN if prev delta ≤ 0|
|`delta_passive_midprice_delta_percent`|`float64`|Change in percent return field|`delta_percent[i] - delta_percent[i-1]`|NaN if first/second bar|
|`delta_passive_midprice_delta_log`|`float64`|Change in log return field|`delta_log[i] - delta_log[i-1]`|NaN if first/second bar|

#### Normalized/CDF Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_passive_midprice_delta_normalized`|`float64`|NaN if first bar|
|`delta_passive_midprice_delta_cdf`|`float64`|NaN if first bar|

#### Volume Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_passive_buy_volume`|`float64`|NaN if first bar|
|`delta_passive_buy_volume_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_passive_buy_volume_log`|`float64`|NaN if first bar or prev ≤ 0|
|`delta_passive_sell_volume`|`float64`|NaN if first bar|
|`delta_passive_sell_volume_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_passive_sell_volume_log`|`float64`|NaN if first bar or prev ≤ 0|

#### Imbalance Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_passive_imbalance_signed`|`float64`|NaN if first bar|
|`delta_passive_imbalance_abs`|`float64`|NaN if first bar|
|`delta_passive_imbalance_signed_ratio`|`float64`|NaN if first bar|
|`delta_passive_imbalance_abs_ratio`|`float64`|NaN if first bar|
|`delta_passive_imbalance_buy_ratio`|`float64`|NaN if first bar|

#### Link Function Transform Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_passive_imbalance_signed_ratio_atanh`|`float64`|NaN if first bar|
|`delta_passive_imbalance_buy_ratio_logit`|`float64`|NaN if first bar|
|`delta_passive_imbalance_abs_ratio_logit`|`float64`|NaN if first bar|

### 11.5 Temporal Deltas

#### Time Elapsed Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_time_elapsed_ns`|`int64`|NaN if first bar|
|`delta_time_elapsed_ns_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_time_elapsed_ns_log`|`float64`|NaN if first bar or prev ≤ 0|

#### Pace of Contracts Traded Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_pace_of_contracts_traded`|`float64`|NaN if first bar|
|`delta_pace_of_contracts_traded_pct`|`float64`|NaN if first bar or prev = 0|
|`delta_pace_of_contracts_traded_log`|`float64`|NaN if first bar or prev ≤ 0|

#### Temporal Log Base Deltas (delta of base log field)

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_time_elapsed_ns_log_base`|`float64`|NaN if first bar|
|`delta_pace_of_contracts_traded_log_base`|`float64`|NaN if first bar|

### 11.6 Divergence Deltas

|Variable|Type|Edge Case Fill|
|---|---|---|
|`delta_divergence_buy_volume`|`float64`|NaN if first bar|
|`delta_divergence_sell_volume`|`float64`|NaN if first bar|
|`delta_divergence_imbalance_signed`|`float64`|NaN if first bar|
|`delta_divergence_imbalance_signed_ratio`|`float64`|NaN if first bar|
|`delta_divergence_buy_ratio`|`float64`|NaN if first bar|

---

## 12. Calculation Order in _calculate_statistics()

```
Step 1: Metadata and Configuration
    - id, bar_volume_size
    - gap_return, max_time_gap_ns, contains_oversized_order, has_resized

Step 2: Order and Volume Statistics (Ambiguous/Total)
    - order_count, order_splits, volume_total, bar_complete

Step 3: Active Metrics
    - Order counts (buy, sell, none)
    - Volumes (buy, sell, none)
    - Imbalances and ratios
    - Link function transforms
    - VWAPs (buy, sell, none)
    - Spread, midpoint
    - Weighted midpoints (imbalance, flow, aggressor)
    - Price ranges (min, max, range for each side)
    - Pace metrics

Step 4: Active N-Side Inferred Classification
    - Determine reference price (with previous bar fallback)
    - Classify N-side trades as inferred buy/sell
    - Calculate inferred volumes and VWAPs

Step 5: Adjusted Metrics (Active + N-Side Inferred)
    - Adjusted volumes
    - Adjusted imbalances and ratios
    - Adjusted link function transforms
    - Adjusted VWAPs
    - Adjusted weighted midpoints

Step 6: Passive Metrics
    - passive_midprice (VWAP of all trades)
    - passive_midprice_delta_price/percent/log
    - passive_midprice_delta_normalized
    - passive_midprice_delta_cdf
    - passive_buy_volume, passive_sell_volume
    - passive imbalances and ratios
    - passive link function transforms

Step 7: Temporal Metrics
    - Timestamps, elapsed time, pace
    - Log transforms

Step 8: Instrument Tracking
    - contract_roll, latest_instrument_id

Step 9: Divergence Metrics (Active vs Passive)
    - Volume divergences
    - Imbalance divergences
    - Ratio divergences

Step 10: Derived Indicator
    - derived_price_direction_positive
```

**Note:** Delta calculations are performed in a separate `_calculate_deltas()` function.

---

## 13. Edge Case Handling Summary

|Scenario|Affected Variables|Fill Value|Rationale|
|---|---|---|---|
|No trades on a side (active)|Side-specific VWAPs, price ranges|NaN|Undefined; missing data|
|No N-side trades|`active_none_*` metrics|0 for counts/volumes, NaN for prices/VWAPs|Count is known (zero); price undefined|
|Entire bar is N-side only|`active_buy_vwap`, `active_sell_vwap`|NaN|No A/B trades; use previous bar fallback for N-side classification|
|Division by zero (ratios)|All ratio metrics|NaN|Mathematically undefined|
|First bar (no previous)|All delta metrics, passive_midprice_delta_*|NaN|No meaningful comparison exists|
|time_elapsed_ns = 0|Pace metrics|NaN|Division by zero|
|σ = 0 from distribution lookup|`passive_midprice_delta_normalized`|NaN|Division by zero|
|passive_midprice_delta_normalized = 0|`passive_midprice_delta_cdf`|0.5|Neutral probability (no directional signal)|
|CDF returns NaN|`passive_buy_volume`, `passive_sell_volume`|0.0|Cannot classify; default to no inferred volume|
|N-side classification with no reference|`active_none_inferred_*`|0.0 for volumes, NaN for VWAPs|Cannot classify without reference price|
|adjusted_volume_buy = 0|`adjusted_buy_vwap`|NaN|No buy volume to weight|
|adjusted_volume_sell = 0|`adjusted_sell_vwap`|NaN|No sell volume to weight|

---

_End of Schema Reference_